# Regresi?n lineal m?ltiple con NumPy

En este notebook vamos a adaptar el ejemplo de regresi?n lineal para una base de datos con **3 variables de entrada**:

- `x1`
- `x2`
- `x3`

Y una variable objetivo:

$$y = 1 + 2x_1 + x_2 + 2x_3$$

La idea es entrenar un modelo para que aprenda una funci?n parecida a esta:

$$h(x) = w_0 + w_1x_1 + w_2x_2 + w_3x_3$$

Donde:

- `w0` es el intercepto o sesgo.
- `w1`, `w2` y `w3` son los pesos de cada variable.

## 1. Importar librer?as

Usaremos principalmente `NumPy`, que permite trabajar con arreglos num?ricos y hacer operaciones matem?ticas de forma eficiente.

Tambi?n usaremos `pandas` solo para mostrar la tabla de datos de manera m?s ordenada.

In [ ]:
import numpy as np
import pandas as pd

## 2. Crear la base de datos

La tabla tiene 7 ejemplos. Cada fila contiene los valores de `x1`, `x2`, `x3` y el resultado esperado `y`.

La f?rmula usada para calcular `y` es:

$$y = 1 + 2x_1 + x_2 + 2x_3$$

In [ ]:
# Datos de entrada
x1 = np.array([1, 2, 3, 4, 5, 6, 7], dtype=np.float32)
x2 = np.array([2, 3, 4, 5, 6, 7, 8], dtype=np.float32)
x3 = np.array([6, 7, 6, 7, 6, 7, 6], dtype=np.float32)

# Variable objetivo
Y = np.array([17, 22, 23, 28, 29, 34, 35], dtype=np.float32)

# Mostrar los datos en una tabla
tabla = pd.DataFrame({
    "x1": x1,
    "x2": x2,
    "x3": x3,
    "y": Y
})

tabla

## 3. Unir las variables de entrada

Para entrenar el modelo, juntamos `x1`, `x2` y `x3` en una sola matriz llamada `X`.

Cada fila representa un ejemplo, y cada columna representa una caracter?stica o variable independiente.

In [ ]:
X = np.column_stack((x1, x2, x3))

print("Matriz X:")
print(X)

print("\nVector Y:")
print(Y)

## 4. Definir el modelo

El modelo de regresi?n lineal m?ltiple ser?:

$$h(x) = w_0 + w_1x_1 + w_2x_2 + w_3x_3$$

Al inicio, los pesos estar?n en cero. Durante el entrenamiento, el algoritmo ir? ajustando esos valores.

In [ ]:
# Pesos iniciales
w0 = 0.0
w1 = 0.0
w2 = 0.0
w3 = 0.0

# Funci?n de predicci?n
def forward(X):
    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = X[:, 2]
    return w0 + w1*x1 + w2*x2 + w3*x3

print("Predicciones iniciales:")
print(forward(X))

## 5. Definir la funci?n de p?rdida

La funci?n de p?rdida mide qu? tan lejos est?n las predicciones del modelo respecto a los valores reales.

Usaremos el error cuadr?tico medio dividido entre 2:

$$J = \frac{1}{2m}\sum (\hat{y} - y)^2$$

Si la p?rdida es grande, el modelo est? prediciendo mal. Si la p?rdida se acerca a cero, el modelo est? aprendiendo bien.

In [ ]:
def loss(y, y_pred):
    return ((y_pred - y) ** 2).mean() / 2

pred_inicial = forward(X)
print("P?rdida inicial:", loss(Y, pred_inicial))

## 6. Calcular gradientes

Los gradientes indican cu?nto debe cambiar cada peso para reducir el error.

Para este modelo tenemos cuatro gradientes:

- Gradiente de `w0`
- Gradiente de `w1`
- Gradiente de `w2`
- Gradiente de `w3`

Luego usaremos esos gradientes en el algoritmo de **descenso por gradiente**.

In [ ]:
def gradient(X, y, y_pred):
    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = X[:, 2]

    error = y_pred - y

    dw0 = error.mean()
    dw1 = (error * x1).mean()
    dw2 = (error * x2).mean()
    dw3 = (error * x3).mean()

    return dw0, dw1, dw2, dw3

## 7. Entrenar el modelo

Ahora entrenamos usando descenso por gradiente.

En cada ?poca el modelo hace lo siguiente:

1. Calcula las predicciones.
2. Calcula la p?rdida.
3. Calcula los gradientes.
4. Actualiza los pesos.

La tasa de aprendizaje (`learning_rate`) controla qu? tan grandes son los cambios en los pesos.

In [ ]:
learning_rate = 0.001
n_iters = 5000

historial_loss = []

for epoch in range(n_iters):
    # 1. Predicci?n
    y_pred = forward(X)

    # 2. P?rdida
    l = loss(Y, y_pred)
    historial_loss.append(l)

    # 3. Gradientes
    dw0, dw1, dw2, dw3 = gradient(X, Y, y_pred)

    # 4. Actualizaci?n de pesos
    w0 -= learning_rate * dw0
    w1 -= learning_rate * dw1
    w2 -= learning_rate * dw2
    w3 -= learning_rate * dw3

    if epoch % 500 == 0:
        print(
            f"epoch {epoch+1}: "
            f"w0={w0:.4f}, w1={w1:.4f}, w2={w2:.4f}, w3={w3:.4f}, "
            f"loss={l:.6f}"
        )

## 8. Ver los pesos aprendidos

La f?rmula real era:

$$y = 1 + 2x_1 + x_2 + 2x_3$$

Por lo tanto, idealmente esperamos algo parecido a:

- `w0 ? 1`
- `w1 ? 2`
- `w2 ? 1`
- `w3 ? 2`

Sin embargo, como `x2` depende directamente de `x1` porque `x2 = x1 + 1`, puede haber m?s de una combinaci?n de pesos que produzca predicciones correctas. Eso se llama **colinealidad**.

In [ ]:
print("Pesos finales aprendidos:")
print(f"w0 = {w0:.4f}")
print(f"w1 = {w1:.4f}")
print(f"w2 = {w2:.4f}")
print(f"w3 = {w3:.4f}")

## 9. Comparar valores reales y predichos

Ahora revisamos si el modelo predice bien los valores de la tabla original.

In [ ]:
y_pred_final = forward(X)

resultados = pd.DataFrame({
    "x1": X[:, 0],
    "x2": X[:, 1],
    "x3": X[:, 2],
    "Y real": Y,
    "Y predicho": y_pred_final,
    "Error": Y - y_pred_final
})

resultados

## 10. Graficar la p?rdida

Esta gr?fica muestra c?mo va disminuyendo el error durante el entrenamiento.

Si la curva baja, significa que el modelo est? aprendiendo.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(historial_loss)
plt.title("Disminuci?n de la p?rdida durante el entrenamiento")
plt.xlabel("?poca")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

## 11. Hacer una nueva predicci?n

Probemos el modelo con un nuevo dato:

- `x1 = 8`
- `x2 = 9`
- `x3 = 7`

Con la f?rmula original:

$$y = 1 + 2(8) + 9 + 2(7) = 40$$

El modelo deber?a predecir un valor cercano a 40.

In [ ]:
nuevo_dato = np.array([[8, 9, 7]], dtype=np.float32)
prediccion = forward(nuevo_dato)

print("Predicci?n para x1=8, x2=9, x3=7:")
print(prediccion[0])

## 12. Conclusi?n

En este notebook construimos una regresi?n lineal m?ltiple desde cero usando NumPy.

El modelo aprendi? a relacionar tres variables de entrada (`x1`, `x2`, `x3`) con una salida `y`.

La funci?n que busc?bamos aproximar era:

$$y = 1 + 2x_1 + x_2 + 2x_3$$

Aunque los pesos finales pueden no ser exactamente iguales a `1`, `2`, `1` y `2`, las predicciones pueden ser muy cercanas debido a que algunas variables est?n relacionadas entre s?.